# Anatomy of a Modern App. 

Open the hood of any modern application and you won't find one program, you'll find a small ecosystem: an API, a background worker, a search index, a message queue, a feature-flag service, all talking to each other over the network. This talk pulls that ecosystem
 apart using a real, working travel app as the specimen. We'll trace a single user action, uploading a photo, as it ripples through five independent services, gets geocoded, tagged by AI, indexed for search, and traced end-to-end, entirely asynchronously. You'll
 leave with a concrete mental model for why modern apps are built this way, what problems that buys you, and what it costs, before we ever get to how Kubernetes holds it all together.

 ---

### Key take aways.   
- Applications are complex.   
- Applications are built for integrations.  
- Require a lot more to manage.   

---

### Summary

A single photo upload, traced end-to-end through every moving part of MyTravels running on Argo CD:

0. **The User Story** 
1. **Upload a photo, watch it ripple** - what happens synchronously in the upload request vs. the four async messages (all sharing one `CorrelationId`) it kicks off.
2. **Address lookup** - `AppendFormattedAddress`, Google Maps/OSM fallback, retry policy, and the 30-minute sweeper.
3. **Where do uploaded images go** - the `uploaded-images` / `resized-images` MinIO buckets and why resizing happens off the critical path.
4. **Integrating an LLM: descriptions and tags** - the `AppendImageTags` → Claude call, the `enable-image-description` flag, and why the Anthropic key lives only on `messaging`.
5. **Enabling search: SOLR** - why every POI read goes through SOLR, how documents converge on one key, and query-time relevance boosts.
6. **Traceability** - how `MessageAuditLogs` and the app-level retry/dead-letter policy let you answer "did it work?" for any correlation id.
7. **What fails first when traffic increases** - per-subscriber serialization, the `AppendImageTags`/Claude bottleneck, and static (non-autoscaled) `messaging` replicas.
8. **Monitoring resources: Prometheus & Grafana** - the "MyTravels Overview" dashboard and the signals behind §7's bottleneck story.
9. **Monitoring the cluster itself: Argo CD** - sync waves, self-heal, and checking the `mytravels` Application's health/sync status live.
10. *(optional live demo)* - forcing a failure by editing the `messaging` deployment's env.
11. **Exposing your data to LLMs** - the `mcp` server's two read-only tools (`search_pointofinterest`, `search_place`), called directly and then fed through Claude with a one-shot example and a JSON Schema-constrained response.
12. *Conclusion** - application architecture has evolved significantly over the past years.

---


### The user story.  

*As a traveller,
I want to upload photos from my trips and have them appear on a map at the location, date, and with a description of what is on a photo,
so that I can visually browse everywhere I've been and when.*

---

#### Understanding system boundaries.   

My Travels will integrate with applications on the internet.

![C4](images/c4-software-system.png)

---

#### The logical architecture.  

One user story - *"as a traveller, I want to upload a photo and see it on a map"* - and it takes
a small mesh of independently-deployable services to deliver it:

![architecture](images/architecture.png)

---

### 1 - Upload a Photo, Watch It Ripple

Open [http://web.mytravels.local:8080](http://web.mytravels.local:8080) and upload a geotagged
photo (or, if it has no GPS EXIF, use the "search for the place instead" flow the upload menu
offers - `POST /api/pointofinterest/image/coordinates`). Come back here once it's done.

**What just happened synchronously**, inside one HTTP request: the original bytes landed in
MinIO's `uploaded-images` bucket, the photo's GPS EXIF was read, and one `PointOfInterests` row
was inserted with a freshly minted `CorrelationId`. That's it - the request returns before any
address, thumbnail, description or search entry exists.

**What happens next, asynchronously:** three messages are published in the same request, all
carrying that one `CorrelationId` - plus a fourth that only fires once one of those three finishes:

| Exchange | Consumer | Fills in |
|---|---|---|
| `append-formatted-address` | `AppendFormattedAddress` | `FormattedAddress` (§2 below) |
| `resize-image` | `ResizeImage` | a thumbnail, then chains `append-image-tags` |
| `append-image-tags` | `AppendImageTags` | description + tags - chained off `resize-image`'s tail, not published at creation (§3-4) |
| `index-solr` | `IndexSolr` | makes the POI findable *right now*, address/description empty |

Five independent services, four separate consumers, zero coordination between them beyond a
shared id. Run the next cell to see the record before any of that has landed.

In [ ]:
%%bash
echo "=== Newest point of interest (fetched right after upload) ==="
curl -s "http://api.mytravels.local:8080/api/pointofinterest?rows=200" | python3 -c "
import json, sys
pois = json.load(sys.stdin)
if not pois:
    print('No points of interest yet - upload one at http://web.mytravels.local:8080 first.')
    sys.exit(0)
p = pois[-1]
print('  id                :', p['id'])
print('  pointOfInterestKey:', p['pointOfInterestKey'])
print('  correlationId     :', p.get('correlationId'))
print('  formattedAddress  :', repr(p.get('formattedAddress')), '(pending geocoding)' if not p.get('formattedAddress') else '')
print('  description       :', repr(p.get('description')), '(pending Anthropic call)' if not p.get('description') else '')
print('  tags              :', p.get('tags'))
"

---

### 2 - Address Lookup

`AppendFormattedAddress` calls a geocoding provider - **Google Maps** if `GoogleApiKey` is a real
key, or a silent, automatic fallback to **OpenStreetMap Nominatim** if it's unset or still the
placeholder. That fallback is a config difference, not a code path anyone has to choose - the app
doesn't know or care which provider answered.

Both providers get the same Polly policy: 2 retries with exponential backoff (~2s, ~4s), no
circuit breaker, no overall timeout. The consumer is idempotent - it skips resolution entirely if
`FormattedAddress` is already non-empty - so an at-least-once redelivery is harmless in the steady
state. A 30-minute sweeper additionally re-scans rows with an empty (not `NULL`) address from the
last 2 days, catching stragglers that missed their first pass.

Watch the same POI pick up its address:

In [ ]:
%%bash
API="http://api.mytravels.local:8080"
ID=$(curl -s "$API/api/pointofinterest?rows=200" | python3 -c "
import json, sys
pois = json.load(sys.stdin)
print(pois[-1]['id'] if pois else '')
")
if [ -z "$ID" ]; then
  echo "No points of interest yet - run the upload cell above first."
  exit 0
fi

echo "=== Polling id=$ID for formattedAddress (up to 60s) ==="
for i in $(seq 1 12); do
  ADDR=$(curl -s "$API/api/pointofinterest?rows=200" | python3 -c "
import json, sys
pois = json.load(sys.stdin)
poi = next((p for p in pois if p['id'] == $ID), None)
print(poi.get('formattedAddress') or '' if poi else '')
")
  if [ -n "$ADDR" ]; then
    echo "  resolved: $ADDR"
    exit 0
  fi
  echo "  attempt $i: still empty..."
  sleep 5
done

echo "No address after 60s - diagnosing inline:"
kubectl get pods -n mytravels-default -l app=messaging -o wide
kubectl logs -n mytravels-default -l app=messaging --tail=60 | grep -i -e address -e geocod || \
  kubectl logs -n mytravels-default -l app=messaging --tail=60

---

### 3 - Where Do Uploaded Images Go?

Two MinIO buckets, both lazily auto-created on first write:

- **`uploaded-images`** - the original, full-resolution file, straight off the upload request.
- **`resized-images`** - a thumbnail at 10% of the original's dimensions, produced by the
  `ResizeImage` consumer.

Why bother resizing at all? The map renders potentially dozens of markers at once - shipping
full-resolution originals to every browser tab would be needless bandwidth and slow paint, so the
expensive resize work happens once, off the critical upload path, in a worker that can be scaled
independently of `api`. `ResizeImage` is idempotent (skips if `ImageResized` is already `true`)
and, on success, chains straight into `append-image-tags` - which is why the AI description in
§4 never fires until the thumbnail exists.

Open the console and browse both buckets live:
[http://minio.mytravels.local:8080](http://minio.mytravels.local:8080)
(`user123` / `password123`) - you'll see one object land in `uploaded-images` immediately and a
second, smaller one appear in `resized-images` a few seconds later.

---

### 4 - Integrating an LLM: Descriptions and Tags

**The app calling an LLM.** Once `ResizeImage` finishes, it chains `append-image-tags`.
`AppendImageTags` checks the `enable-image-description` Flagsmith flag
([http://flagsmith.mytravels.local:8080](http://flagsmith.mytravels.local:8080), default `true`,
fails open if Flagsmith is unreachable) and, if it's on, sends the **original** image to Claude -
`AnthropicImageDescriptionService`, structured output pinned to `{ description, tags[] }`, model
`claude-haiku-4-5` by default - and persists both fields. The Anthropic key lives **only** on
`messaging`; `api` and `mcp` never see it, because they never make the call. Turning the flag off
is a clean way to run this whole stack with no LLM key configured at all: uploads, addresses and
thumbnails still work, only the description/tags step is skipped - and `index-solr` still fires
either way, so the POI stays searchable.


In [ ]:
%%bash
API="http://api.mytravels.local:8080"
ID=$(curl -s "$API/api/pointofinterest?rows=200" | python3 -c "
import json, sys
pois = json.load(sys.stdin)
print(pois[-1]['id'] if pois else '')
")
if [ -z "$ID" ]; then
  echo "No points of interest yet - run the upload cell above first."
  exit 0
fi

echo "=== Polling id=$ID for description/tags (calls Claude - can take up to a minute) ==="
for i in $(seq 1 12); do
  RESULT=$(curl -s "$API/api/pointofinterest?rows=200" | python3 -c "
import json, sys
pois = json.load(sys.stdin)
poi = next((p for p in pois if p['id'] == $ID), None)
if poi and poi.get('description'):
    print(f\"description={poi['description']!r}\")
    print('tags=' + ', '.join(t['name'] for t in poi.get('tags') or []))
"
  )
  if [ -n "$RESULT" ]; then
    echo "$RESULT"
    exit 0
  fi
  echo "  attempt $i: nothing yet..."
  sleep 5
done

echo "No description after 60s - this is normal if enable-image-description is off, or if"
echo "ANTHROPIC_API_KEY is a placeholder. Diagnosing inline:"
kubectl logs -n mytravels-default -l app=messaging --tail=60 | grep -i -e AppendImageTags -e anthropic || \
  kubectl logs -n mytravels-default -l app=messaging --tail=60

---

### 5 - Enabling Search: SOLR

Since `api:v1.0.12`, **every POI read goes through SOLR** - the map's own list
(`GET /api/pointofinterest`), free-text search (`GET /api/pointofinterest/search`), and the MCP
tool above. PostgreSQL's `ILIKE` search is gone; SOLR is a *derived* store, rebuildable from
Postgres at any time via `POST /api/pointofinterest/reindex`.

Two things make this async pipeline converge instead of racing itself:

- **Documents are keyed on `PointOfInterestKey`, not the row id** - replacing a photo re-uses the
  key, so a rebuild and incremental indexing always agree on one document per key.
- **`index-solr` fires three times per upload** (creation, then the tail of address resolution,
  then the tail of tagging) because those fields all arrive on different timelines and SOLR
  upserts by key - repeat indexing is free.

Relevance boosts (`formatted_address^5 tags^3 description^1`) are query-time, so retuning ranking
needs no reindex. `rows` defaults to 100 and is silently truncated past that on both the list and
`/search` - the accepted cost of one capped query instead of a paging loop.

**Running the same query straight against SOLR** - useful to see the relevance scoring `/search`
is built on, with `api` out of the loop entirely. `SolrSearchService` sends the same `defType`,
`qf`, `rows` and `start` on every call - fill them into the
[SOLR admin console](http://solr.mytravels.local:8080/solr/#/mytravels-pois/query)
(Core Selector → `mytravels-pois` → Query) instead:

| Field | Value |
|---|---|
| `q` | `landscape` |
| `defType` | `edismax` |
| `qf` | `formatted_address^5 tags^3 description^1` |
| `rows` | `50` |
| `start` | `0` |
| `wt` | `json` |

Swap `q` for `city` or `trees` to reproduce the other two rows the next cell prints.

In [ ]:
%%bash
SOLR="http://solr.mytravels.local:8080"
API="http://api.mytravels.local:8080"

INDEXED=$(curl -s "$SOLR/solr/mytravels-pois/select?q=*:*&rows=0" | python3 -c "
import json, sys
print(json.load(sys.stdin)['response']['numFound'])
" 2>/dev/null || echo "?")
KEYS=$(curl -s "$API/api/pointofinterest?rows=200" | python3 -c "
import json, sys
print(len({p['pointOfInterestKey'] for p in json.load(sys.stdin)}))
")
echo "=== SOLR documents: $INDEXED   |   distinct keys via the API: $KEYS ==="

echo ""
echo "=== Free-text relevance search (address ^5, tags ^3, description ^1) ==="
for TERM in landscape city trees; do
  N=$(curl -s "$API/api/pointofinterest/search?term=$TERM&rows=50" | python3 -c "
import json, sys
print(len(json.load(sys.stdin)))
" 2>/dev/null || echo 0)
  printf "  term=%-10s %s result(s)\n" "$TERM" "$N"
done

---

### 6 - How Do You Know When a Message Fails? Traceability

To simulate a failed message disable a key [Claude Conole](https://platform.claude.com/dashboard)

Every message above carried the same `CorrelationId`. `MessagePublisher` writes a `Published` row
before every publish; `MessageSubscriberBase<T>` writes `ConsumeSucceeded`, `Retried`, or `Failed`
after every consume attempt - and that audit logging is wrapped in its own try/catch, so a logging
failure can never block the pipeline it's describing (the inverse also holds: a gap in the
timeline is not proof a step didn't run).

**The retry policy lives in application code, not the broker** - there is no dead-letter exchange,
no TTL, no max-length on any queue. On failure, `MessageSubscriberBase<T>` republishes to the
*same* exchange with an incremented `x-retry-count` header, immediately, no backoff. On the fourth
attempt it publishes a `FailedMessage` to the paired `<exchange>-failed` exchange and acks the
original - but **nothing is bound to those `-failed` exchanges**, so the message itself is
discarded on arrival. The `Failed` audit row is the only durable trace that it ever happened.

Read side: [http://web.mytravels.local:8080/traceability](http://web.mytravels.local:8080/traceability),
or `GET /api/traceability` / `GET /api/traceability/{id}` directly - both gated behind the
`enable-message-tracing` flag (writes are unaffected; only the read side 404s when it's off).

Pull the full timeline for the upload from §1:

In [ ]:
%%bash
API="http://api.mytravels.local:8080"
CORR=$(curl -s "$API/api/pointofinterest?rows=200" | python3 -c "
import json, sys
pois = json.load(sys.stdin)
print(pois[-1].get('correlationId') or '' if pois else '')
")
if [ -z "$CORR" ]; then
  echo "No correlation id available - run the upload cell in §1 first."
  exit 0
fi

echo "=== Event timeline for correlationId=$CORR ==="
curl -s "$API/api/traceability/$CORR" | python3 -c "
import json, sys
events = json.load(sys.stdin)
if not events:
    print('  (no audit rows yet - the pipeline may still be in flight)')
for e in events:
    marker = '  FAILED  ' if e['eventType'] == 'Failed' else ('  retried ' if e['eventType'] == 'Retried' else '          ')
    print(f\"{marker}{e['createdAt']}  {e['exchangeName']:<28} {e['eventType']:<16} {e.get('errorMessage') or ''}\")
"
echo ""
echo "That's one photo upload, traced end-to-end across every service that touched it."

---

### 7 - What Fails First When Traffic Increases?

RabbitMQ is the shock absorber - an upload burst just makes queues deeper, not the API slower.
But depth isn't free capacity; three things bound how fast that depth drains:

- **Each subscriber class serializes itself** behind a `static SemaphoreSlim(1,1)`, so
  `AppendFormattedAddress`, `ResizeImage`, and `AppendImageTags` each process **one message at a
  time per process**, regardless of RabbitMQ's `prefetchCount: 10`. The three pipelines run in
  parallel *with each other*; each one alone is strictly serial.
- **`AppendImageTags` is the tightest of the three** - it's not just serialized, it's also waiting
  on a metered third-party call (Claude) every time, so its queue is the one that visibly backs up
  first under load.
- **`messaging` runs 2 static replicas, with no autoscaler anywhere in these manifests** - so
  under sustained load, the fix is a manual `kubectl scale`, not a HorizontalPodAutoscaler
  reacting to queue depth.

One more compounding factor on the producer side: `MessagePublisher` opens a brand-new AMQP
connection and channel **per publish call** - no pooling - so the cost of publishing itself grows
with traffic too, on top of the consumer-side serialization.

Watch queue depth live: [http://rabbitmq.mytravels.local:8080](http://rabbitmq.mytravels.local:8080)
(`user123` / `password123`), or pull it here:

In [ ]:
%%bash
# Point PHOTOS_DIR at a folder of geotagged photos.
# The cell skips itself if the folder does not exist, so it is safe to run top-to-bottom.
PHOTOS_DIR="${PHOTOS_DIR:-$HOME/Personal/photos}"
API_BASE="http://api.mytravels.local:8080"

if [ ! -d "$PHOTOS_DIR" ]; then
  echo "SKIPPED - not a directory: $PHOTOS_DIR"
  echo "Set PHOTOS_DIR above to a folder of geotagged photos and re-run this cell to seed data."
  exit 0
fi

if ! curl -s -o /dev/null -m 5 "$API_BASE/api/pointofinterest"; then
  echo "API not reachable at $API_BASE - diagnosing inline:"
  echo "--- Application (a failed sync looks like a down API) ---"
  kubectl get application mytravels -n argocd
  kubectl get application mytravels -n argocd \
    -o jsonpath='{.status.operationState.phase}: {.status.operationState.message}{"\n"}'
  echo "--- pods (app=api) ---"
  kubectl get pods -n mytravels-default -l app=api -o wide
  echo "--- api logs (last 30) ---"
  kubectl logs -n mytravels-default -l app=api --tail=30
  exit 1
fi

echo "=== Uploading photos from: $PHOTOS_DIR ==="
../.claude/scripts/upload-photos.sh "$PHOTOS_DIR" "$API_BASE"

In [ ]:
%%bash
RMQ="http://rabbitmq.mytravels.local:8080"
USER=$(grep -E '^RABBITMQ_DEFAULT_USER=' .env | cut -d= -f2-)
PASS=$(grep -E '^RABBITMQ_DEFAULT_PASS=' .env | cut -d= -f2-)

echo "=== Queue depth per pipeline (work queues, named after their exchange) ==="
curl -s -u "$USER:$PASS" "$RMQ/api/queues" | python3 -c "
import json, sys
queues = json.load(sys.stdin)
for q in sorted(queues, key=lambda q: q['name']):
    print(f\"  {q['name']:<28} ready={q.get('messages_ready', 0):<5} unacked={q.get('messages_unacknowledged', 0):<5} consumers={q.get('consumers', 0)}\")
" 2>/dev/null || echo "Could not reach the management API - check .env has RABBITMQ_DEFAULT_USER/PASS."

echo ""
echo "=== messaging replica count (static - no HPA in these manifests) ==="
kubectl get deployment messaging -n mytravels-default -o jsonpath='replicas: {.spec.replicas}{"\n"}'

---

### 8 - Monitor Your Resources: Prometheus & Grafana

Every application service exports OTLP traces and metrics; `postgres-exporter` and `cAdvisor` add
database and container-level metrics; Prometheus scrapes all of it; Grafana renders the
**"MyTravels Overview"** dashboard - which happens to be exactly the signals from §7's bottleneck
story, in one place:

- API request rate
- API error rate (5xx)
- **RabbitMQ queue depth** - the same numbers as the cell above, over time
- Postgres active connections
- Container CPU usage

[http://grafana.mytravels.local:8080](http://grafana.mytravels.local:8080) ·
[http://prometheus.mytravels.local:8080](http://prometheus.mytravels.local:8080)
(`user123` / `password123`)

In [ ]:
%%bash
echo "=== Prometheus scrape targets ==="
curl -s http://prometheus.mytravels.local:8080/api/v1/targets \
  | python3 -c "
import json, sys
data = json.load(sys.stdin)
for t in data['data']['activeTargets']:
    print(f\"  {t['labels'].get('job'):<20} {t['health']:<8} {t.get('lastError', '')}\")
"

---

### 9 - Monitoring the Cluster Itself: Argo CD

Everything above is only running because Argo CD continuously reconciles the cluster against
`manifests/` on git - sync waves order the dependencies (namespace → secrets → postgres →
migrations → everything else), and self-heal reverts manual drift on its own. That mechanism is
the subject of the *next* part of this talk (Kubernetes); here it's worth showing once, live,
because it's the same "who's watching this system" question as §6 and §8, aimed at the platform
instead of the app.

[http://argocd.mytravels.local:8080](http://argocd.mytravels.local:8080)

![k8scluster](images/k8s-components.png)


In [ ]:
%%bash
echo "=== Application health/sync, as Argo sees it right now ==="
kubectl get application mytravels -n argocd \
  -o jsonpath='sync={.status.sync.status}  health={.status.health.status}{"\n"}'
echo ""
echo "=== Resources it manages ==="
kubectl get application mytravels -n argocd \
  -o jsonpath='{range .status.resources[*]}{.kind}{"/"}{.name}{" "}{.status}{"\n"}{end}'

#### Optional live demo - force a failure

In [ ]:
%%bash
kubectl set env deployment/messaging -n mytravels-default DOTNET_hostBuilder__reloadConfigOnChange=false

---

### 11 - Exposing your data to LLMs

**The LLM calling the app.** `mcp` exposes the *same* domain services as two read-only MCP tools
over streamable HTTP - no REST surface, no Swagger, its own ingress host - so an MCP-aware LLM
client (Claude Desktop, an agent) can query this data directly:

| Tool | Arguments | Backed by |
|---|---|---|
| `search_pointofinterest` | `term` (not `query`) | the same SOLR index as `/search` |
| `search_place` | `query`, `limit` | the same maps service as `GET /api/place` |

It's the same asymmetry as the description feature, mirrored: one side of the app talks *to* an
LLM, the other side *is* a tool an LLM talks to - and both are anonymous, unauthenticated, and
entirely separate code paths that happen to share the same domain services underneath.

In [ ]:
%%bash
echo "=== mcp health (its only plain-HTTP route - everything else is MCP-over-HTTP) ==="
curl -s http://mcp.mytravels.local:8080/health
echo ""

**Calling the tool directly.** No client library needed - `mcp` runs `WithHttpTransport()`
in stateless mode here (no `Mcp-Session-Id` handshake to track), so a single JSON-RPC POST is
a complete round trip: `tools/call` with
`{"name": "search_pointofinterest", "arguments": {"term": "..."}}` against
`http://mcp.mytravels.local:8080/`. The `content[0].text` in the reply is the same JSON list
`PointOfInterestService`/`SolrSearchService` hand back to `api` - same domain service, same
SOLR index, just a different transport in front of it.

**Handing that JSON to an LLM.** Anthropic's servers can't reach `mcp.mytravels.local` - it's
a hostname that only resolves on this machine - so the MCP *connector* (Claude calling the
tool itself, server-side) has nothing to connect to here. What follows instead is one Messages
API call that reuses the same `ANTHROPIC_API_KEY`/`ANTHROPIC_MODEL` `messaging` already has in
`.env` for §4. The prompt pairs one worked example (a one-shot: a sample term, sample raw hits,
and the JSON it should produce) with `output_config.format` - a JSON Schema the response is
constrained to match - so what comes back below is exactly `{ query, result_count, results[] }`,
never prose wrapped around it.

In [ ]:
%%bash
API="http://api.mytravels.local:8080"
MCP="http://mcp.mytravels.local:8080/"

TERM="malalane"

echo "=== 1. tools/call search_pointofinterest term=\"$TERM\" ==="
MCP_RESULTS=$(curl -s -m 10 -X POST "$MCP" \
  -H "Content-Type: application/json" \
  -H "Accept: application/json, text/event-stream" \
  -d "{\"jsonrpc\":\"2.0\",\"id\":1,\"method\":\"tools/call\",\"params\":{\"name\":\"search_pointofinterest\",\"arguments\":{\"term\":\"$TERM\"}}}" \
  | sed -n 's/^data: //p' \
  | python3 -c "import json,sys; print(json.loads(sys.stdin.read())['result']['content'][0]['text'])")
echo "$MCP_RESULTS" | python3 -m json.tool

echo ""
echo "=== 2. Claude turns those hits into structured JSON (one-shot example + output_config.format) ==="
ANTHROPIC_API_KEY=$(grep -E '^ANTHROPIC_API_KEY=' .env | cut -d= -f2-)
ANTHROPIC_MODEL=$(grep -E '^ANTHROPIC_MODEL=' .env | cut -d= -f2-)
if [ -z "$ANTHROPIC_API_KEY" ]; then
  echo "ANTHROPIC_API_KEY is not set in .env - skipping the LLM call."
  exit 0
fi

REQUEST_BODY=$(python3 - "$TERM" "$MCP_RESULTS" "$ANTHROPIC_MODEL" << 'PY'
import json, sys
term, mcp_results, model = sys.argv[1], sys.argv[2], sys.argv[3] or "claude-haiku-4-5"

schema = {
    "type": "object",
    "properties": {
        "query": {"type": "string"},
        "result_count": {"type": "integer"},
        "results": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "poi_id": {"type": "integer"},
                    "address": {"type": "string"},
                    "tags": {"type": "array", "items": {"type": "string"}},
                    "why_it_matched": {"type": "string"}
                },
                "required": ["poi_id", "address", "tags", "why_it_matched"],
                "additionalProperties": False
            }
        }
    },
    "required": ["query", "result_count", "results"],
    "additionalProperties": False
}

system_prompt = (
    "You summarize raw JSON search hits from the MyTravels `search_pointofinterest` MCP tool "
    "into a compact structured report for a travel app's chat UI. Only use the data given to "
    "you - never invent a POI, address, or tag that isn't in the input."
)

one_shot_user = (
    'search term: "beach"\n'
    'raw MCP results: '
    '[{"id":42,"formattedAddress":"Clifton 4th Beach, Cape Town, South Africa",'
    '"tags":[{"id":9,"name":"sunset"},{"id":10,"name":"beach"}]}]'
)
one_shot_assistant = json.dumps({
    "query": "beach",
    "result_count": 1,
    "results": [{
        "poi_id": 42,
        "address": "Clifton 4th Beach, Cape Town, South Africa",
        "tags": ["sunset", "beach"],
        "why_it_matched": "address and tags both mention beach"
    }]
})

real_user = f'search term: "{term}"\nraw MCP results: {mcp_results}'

body = {
    "model": model,
    "max_tokens": 1024,
    "system": system_prompt,
    "messages": [
        {"role": "user", "content": one_shot_user},
        {"role": "assistant", "content": one_shot_assistant},
        {"role": "user", "content": real_user}
    ],
    "output_config": {"format": {"type": "json_schema", "schema": schema}}
}
print(json.dumps(body))
PY
)

RESPONSE=$(curl -s -m 30 https://api.anthropic.com/v1/messages \
  -H "Content-Type: application/json" \
  -H "x-api-key: $ANTHROPIC_API_KEY" \
  -H "anthropic-version: 2023-06-01" \
  -d "$REQUEST_BODY")

echo "$RESPONSE" | python3 -c "
import json, sys
r = json.loads(sys.stdin.read())
if 'error' in r:
    print('Anthropic API error:', json.dumps(r['error'], indent=2))
else:
    text = next(b['text'] for b in r['content'] if b['type'] == 'text')
    print(json.dumps(json.loads(text), indent=2))
"

---

### 12 - Conclusion

![the evolution](images/the-evolution.png)

Application architecture has evolved significantly over the past few decades, moving from tightly coupled monolithic applications to increasingly distributed and loosely coupled systems. We moved from monoliths, where functionality and data lived within a single application, to Service-Oriented Architecture (SOA), where capabilities were exposed as independent services and integrated through technologies such as SOAP and later REST. This evolved further into microservices, where smaller, independently deployable services communicate through APIs and asynchronous messaging using technologies such as message brokers, AMQP, and gRPC.

We are now entering another stage of this evolution, where applications are not only integrating with other applications and services, but also with AI agents. Protocols such as the Model Context Protocol (MCP) provide a standardised way for agents to discover and interact with tools, data, and services. In many ways, this represents another shift in application integration: from applications calling services, to agents dynamically discovering and using capabilities on behalf of users.
